In [0]:
# Create catalog
spark.sql("CREATE CATALOG IF NOT EXISTS finguard")

# Create schemas under the catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS finguard.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS finguard.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS finguard.gold")

print("Catalog 'finguard' created successfully")
print("Schemas created: bronze, silver, gold")

In [0]:
# Create source schema under finguard catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS finguard.source")
print("Schema 'finguard.source' created successfully")

# Create transactions and fraud_watchlist volume under source schema
spark.sql("CREATE VOLUME IF NOT EXISTS finguard.source.transactions")
print("Volume 'finguard.source.transactions' created successfully")

spark.sql("CREATE VOLUME IF NOT EXISTS finguard.source.fraud_watchlist")
print("Volume 'finguard.source.fraud_watchlist' created successfully")

# Create checkpoint directory within the transactions volume
dbutils.fs.mkdirs("/Volumes/finguard/source/transactions/checkpoint")
print("Directory 'checkpoint' created successfully at /Volumes/finguard/source/transactions/checkpoint")

In [0]:
# bootstrap_server="pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092"
# api_key="CIHL6TYYZ2K7A47A"
# api_secret='cflt4+7yh9ShdfqbcybRKEml/MOUX9iFAktWbKn8Ju8oxfBXKy2c3AnawngueYZg'
# topics='credit_card-transaction'


In [0]:
import json
kafka_connection_json=dbutils.secrets.get(scope="finguard-scope",key="kafka_connection_details")
kafka_config=json.loads(kafka_connection_json)
bootstrap_servers=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic=kafka_config['topic']

In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
sample_batch = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",jaas_config)
    .option("startingOffsets","earliest")
    .load()
)

In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

In [0]:
from pyspark.sql.functions import col
parsed_batch = sample_batch.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    # from_json(col("value").cast("string"), schema).alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)


In [0]:
display(parsed_batch)

In [0]:
parsed_batch.write.mode("ignore").saveAsTable("finguard.bronze.transactions_batch_test")

In [0]:
streaming_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",jaas_config)
    .option("startingOffsets","earliest")
    .load()
)

In [0]:
# from pyspark.sql.functions import col
parsed_streaming_df = streaming_df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    # from_json(col("value").cast("string"), schema).alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)


In [0]:
steaming_query=(parsed_streaming_df.writeStream.format("delta")
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/finguard/source/transactions/checkpoint/")
 .trigger(availableNow=True)
 .toTable("finguard.bronze.transactions_streaming_test")
)

print("Query started with query id: ",steaming_query.id)

In [0]:
%sql
select * from finguard.bronze.transactions_streaming_test